In [ ]:
!pip install ollama pymupdf

In [20]:
import fitz
import ollama

In [21]:
def ask (question:str) -> str:
    response = ollama.chat(
        model = "llama3.2",
        messages=[{"role":"user", "content":question}]
    )
    return response["message"]["content"]

In [22]:
print(ask("""Questions:
1. What is the capital city of Bangladesh?
2. Which river is considered the lifeline of Bangladesh?
3. In which year did Bangladesh gain independence?
4. What is the official language of Bangladesh?
5. What is the name of the world’s largest mangrove forest located in Bangladesh?
6. Which currency is used in Bangladesh? What is the national animal of Bangladesh?"""
))

Here are the answers to your questions:

1. The capital city of Bangladesh is Dhaka.
2. The river considered the lifeline of Bangladesh is the Padma River (also known as the Ganges River).
3. Bangladesh gained independence on December 16, 1971.
4. The official language of Bangladesh is Bengali.
5. The world's largest mangrove forest located in Bangladesh is the Sundarbans Mangrove Forest.
6. The currency used in Bangladesh is the Bangladeshi Taka (BDT). The national animal of Bangladesh is the Bengal Tiger.


In [23]:
def extract_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    return "\n".join(page.get_text("text") for page in doc)

In [24]:
def chunk_text(text: str, chunk_size: int = 400) -> list:
    words = text.split()
    return ["".join(words[i: i+chunk_size]) for i in range(0, len(words), chunk_size)]

In [25]:
def ask_pdf(pdf_path:str, question:str) -> str:
    text = extract_text(pdf_path)
    chunks = chunk_text(text)

    for chunk in chunks:
        response = ollama.chat(
            model = "llama3.2",
            messages = [{
                "role": "user",
                "content": f"Document:\n{chunk}\n\nQuestion(s): {question}\n\nIf the answer to any question is not found in document, reply with 'Answer not found in doc' for that question only."
            }])
        answer = response["message"]["content"]
        if "not found" not in answer.lower():
            return answer
            
    return "Answer not found in the document"

In [33]:
import re

parts = re.split(r'\d+\.\s+', """1. What is the capital city of Bangladesh?
                                 2. Which river is considered the lifeline of Bangladesh?
                                 3. In which year did Bangladesh gain independence?
                                 4. What is the official language of Bangladesh?
                                 5. What is the name of the world’s largest mangrove forest located in Bangladesh?
                                 6. Which currency is used in Bangladesh? What is the national animal of Bangladesh?""")
parts = [p.strip() for p in parts if p.strip()]

questions_list = []
for part in parts:
    matches = re.findall(r'[^?]+\?', part)
    questions_list.extend([m.strip() for m in matches])

print(questions_list)

['What is the capital city of Bangladesh?', 'Which river is considered the lifeline of Bangladesh?', 'In which year did Bangladesh gain independence?', 'What is the official language of Bangladesh?', 'What is the name of the world’s largest mangrove forest located in Bangladesh?', 'Which currency is used in Bangladesh?', 'What is the national animal of Bangladesh?']


In [34]:
answers = [ask_pdf("Test PDF (Hard).pdf", q) for q in questions_list]

for i, ans in enumerate(answers, 1):
    print(f"Answer {i}: {ans}")

Answer 1: Dhaka.
Answer 2: The Padma River is often referred to as the lifeline of Bangladesh.
Answer 3: Bangladesh gained its independence in 1971.
Answer 4: The official language of Bangladesh is Bengali (also known as Bangla).
Answer 5: The Sundarbans.
Answer 6: The Bangladeshi Taka.
Answer 7: The Royal Bengal Tiger.
